In [2]:
import gzip

vcf_path = r"D:\variant_data\GCF_000001405.40.gz"

with gzip.open(vcf_path, "rt", encoding="utf-8") as f:

    for line in f:

        if line.startswith("##INFO"):

            print(line.strip())

        elif line.startswith("#CHROM"):
            break

##INFO=<ID=RS,Number=1,Type=Integer,Description="dbSNP ID (i.e. rs number)">
##INFO=<ID=GENEINFO,Number=1,Type=String,Description="Pairs each of gene symbol:gene id.  The gene symbol and id are delimited by a colon (:) and each pair is delimited by a vertical bar (|).  Does not include pseudogenes.">
##INFO=<ID=PSEUDOGENEINFO,Number=1,Type=String,Description="Pairs each of pseudogene symbol:gene id.  The pseudogene symbol and id are delimited by a colon (:) and each pair is delimited by a vertical bar (|)">
##INFO=<ID=dbSNPBuildID,Number=1,Type=Integer,Description="First dbSNP Build for RS (NOTE: absent for RS created from Frequency Data without Submission SNPs)">
##INFO=<ID=SAO,Number=1,Type=Integer,Description="Variant Allele Origin: 0 - unspecified, 1 - Germline, 2 - Somatic, 3 - Both">
##INFO=<ID=SSR,Number=1,Type=Integer,Description="Variant Suspect Reason Codes (may be more than one value added together) 0 - unspecified, 1 - Paralog, 2 - byEST, 4 - oldAlign, 8 - Para_EST, 16 - 1k

In [ ]:
import gzip

vcf_path = r"D:\variant_data\GCF_000001405.40.gz"

BENIGN = {"2", "3", "15"}
PATHOGENIC = {"4", "5", "16"}


def parse_info(info_string):
    info = {}

    for item in info_string.split(";"):

        if "=" in item:
            key, value = item.split("=", 1)
            info[key] = value

        else:
            # VCF flag, ví dụ NSM
            info[item] = True

    return info


count = 0
missense_count = 0
clinical_count = 0


with gzip.open(vcf_path, "rt", encoding="utf-8") as f:

    for line in f:

        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        chrom = fields[0]
        pos = fields[1]
        rsid = fields[2]
        ref = fields[3]
        alt = fields[4]
        info = parse_info(fields[7])

        # 1. Chỉ missense
        if "NSM" not in info:
            continue

        missense_count += 1

        # 2. Phải có clinical significance
        if "CLNSIG" not in info:
            continue

        clinical_count += 1

        print("=" * 80)
        print("RSID:", rsid)
        print("CHROM:", chrom)
        print("POS:", pos)
        print("REF:", ref)
        print("ALT:", alt)
        print("GENE:", info.get("GENEINFO"))
        print("CLNSIG:", info.get("CLNSIG"))
        print("CLNHGVS:", info.get("CLNHGVS"))
        print("CLNDN:", info.get("CLNDN"))
        print("CLNREVSTAT:", info.get("CLNREVSTAT"))
        print("CLNACC:", info.get("CLNACC"))

        count += 1

        if count >= 20:
            break


print("\nMissense:", missense_count)
print("Missense + clinical:", clinical_count)

RSID: rs781394307
CHROM: NC_000001.11
POS: 69134
REF: A
ALT: G
GENE: OR4F5:79501
CLNSIG: .,3
CLNHGVS: NC_000001.11:g.69134=,NC_000001.11:g.69134A>G
CLNDN: .,not_specified
CLNREVSTAT: .,single
CLNACC: .,RCV004071174.1
RSID: rs2521653848
CHROM: NC_000001.11
POS: 69314
REF: T
ALT: G
GENE: OR4F5:79501
CLNSIG: .,0
CLNHGVS: NC_000001.11:g.69314=,NC_000001.11:g.69314T>G
CLNDN: .,not_specified
CLNREVSTAT: .,single
CLNACC: .,RCV004496909.1
RSID: rs2521654033
CHROM: NC_000001.11
POS: 69423
REF: G
ALT: A
GENE: OR4F5:79501
CLNSIG: .,0
CLNHGVS: NC_000001.11:g.69423=,NC_000001.11:g.69423G>A
CLNDN: .,not_specified
CLNREVSTAT: .,single
CLNACC: .,RCV004496910.1
RSID: rs1570409925
CHROM: NC_000001.11
POS: 69581
REF: C
ALT: G
GENE: OR4F5:79501
CLNSIG: .,0
CLNHGVS: NC_000001.11:g.69581=,NC_000001.11:g.69581C>G
CLNDN: .,not_specified
CLNREVSTAT: .,single
CLNACC: .,RCV004109876.1
RSID: rs766444643
CHROM: NC_000001.11
POS: 69682
REF: G
ALT: A
GENE: OR4F5:79501
CLNSIG: .,0
CLNHGVS: NC_000001.11:g.69682=,NC_00

In [ ]:
import gzip
import os
import re
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

VCF_PATH = r"D:\\variant_data\\dbsnp_GCF_000001405.40.gz"

OUTPUT_PATH = r"D:\\variant_data\\dbsnp_missense_raw.parquet"

# Number of output rows accumulated before writing
BATCH_SIZE = 100_000


# ============================================================
# CLNSIG DEFINITIONS
# ============================================================

BENIGN_CODES = {
    "2",   # Benign
    "3",   # Likely benign
    "15"   # Benign/Likely benign
}

PATHOGENIC_CODES = {
    "4",   # Likely pathogenic
    "5",   # Pathogenic
    "16"   # Pathogenic/Likely pathogenic
}


# ============================================================
# CHROMOSOME CONVERSION
# ============================================================

REFSEQ_TO_CHR = {
    "NC_000001.11": "1",
    "NC_000002.12": "2",
    "NC_000003.12": "3",
    "NC_000004.12": "4",
    "NC_000005.10": "5",
    "NC_000006.12": "6",
    "NC_000007.14": "7",
    "NC_000008.11": "8",
    "NC_000009.12": "9",
    "NC_000010.11": "10",
    "NC_000011.10": "11",
    "NC_000012.12": "12",
    "NC_000013.11": "13",
    "NC_000014.9": "14",
    "NC_000015.10": "15",
    "NC_000016.10": "16",
    "NC_000017.11": "17",
    "NC_000018.10": "18",
    "NC_000019.10": "19",
    "NC_000020.11": "20",
    "NC_000021.9": "21",
    "NC_000022.11": "22",
    "NC_000023.11": "X",
    "NC_000024.10": "Y",
}


def convert_chrom(chrom):
    """
    Convert RefSeq chromosome accession to chromosome number.
    """

    return REFSEQ_TO_CHR.get(chrom, chrom)


# ============================================================
# PARSE INFO
# ============================================================

def parse_info(info_string):

    info = {}

    for item in info_string.split(";"):

        if "=" in item:
            key, value = item.split("=", 1)
            info[key] = value

        else:
            # VCF flag
            info[item] = True

    return info


# ============================================================
# GET ALLELE-SPECIFIC VALUES
# ============================================================

def get_allele_values(info, key, n_alt):

    expected = n_alt + 1

    if key not in info:
        return ["."] * expected

    values = info[key].split(",")

    if len(values) < expected:
        values += ["."] * (expected - len(values))

    return values[:expected]


# ============================================================
# GET BINARY LABEL
# ============================================================

def get_label(clnsig):

    if clnsig == ".":
        return None

    # Multiple assertions for the same allele
    values = set(clnsig.split("|"))

    values.discard(".")

    if not values:
        return None

    # Conflicting benign + pathogenic
    if (
        values & BENIGN_CODES
        and values & PATHOGENIC_CODES
    ):
        return None

    # Benign / likely benign
    if values.issubset(BENIGN_CODES):
        return 0

    # Pathogenic / likely pathogenic
    if values.issubset(PATHOGENIC_CODES):
        return 1

    return None


# ============================================================
# OUTPUT SCHEMA
# ============================================================

schema = pa.schema([
    ("Variant_ID", pa.string()),
    ("rsid", pa.string()),
    ("chrom", pa.string()),
    ("pos", pa.int64()),
    ("ref", pa.string()),
    ("alt", pa.string()),
    ("gene", pa.string()),
    ("clnsig", pa.string()),
    ("label", pa.int8()),
    ("clnhgvs", pa.string()),
    ("clndn", pa.string()),
    ("clnrevstat", pa.string()),
    ("clnacc", pa.string()),
])


# ============================================================
# REMOVE OLD OUTPUT
# ============================================================

if os.path.exists(OUTPUT_PATH):

    print("Output file already exists:")
    print(OUTPUT_PATH)

    answer = input(
        "Delete and recreate it? [y/n]: "
    ).strip().lower()

    if answer == "y":
        os.remove(OUTPUT_PATH)
    else:
        raise SystemExit(
            "Stopped. Choose another OUTPUT_PATH."
        )


# ============================================================
# STATISTICS
# ============================================================

total_records = 0
nsm_records = 0
output_rows = 0

benign_count = 0
pathogenic_count = 0
excluded_count = 0

chrom_counts = {}


# ============================================================
# PARQUET WRITER
# ============================================================

writer = pq.ParquetWriter(
    OUTPUT_PATH,
    schema,
    compression="zstd"
)


batch = []


def write_batch():

    global batch

    if not batch:
        return

    table = pa.Table.from_pylist(
        batch,
        schema=schema
    )

    writer.write_table(table)

    batch = []


# ============================================================
# PROCESS VCF
# ============================================================

print("=" * 70)
print("Starting dbSNP extraction")
print("=" * 70)

print("Input:")
print(VCF_PATH)

print("\nOutput:")
print(OUTPUT_PATH)

print("\n")


try:

    with gzip.open(
        VCF_PATH,
        "rt",
        encoding="utf-8"
    ) as vcf:

        for line in tqdm(
            vcf,
            desc="Processing dbSNP",
            unit=" records"
        ):

            if line.startswith("#"):
                continue

            total_records += 1

            fields = line.rstrip("\n").split("\t")

            if len(fields) < 8:
                continue

            chrom_raw = fields[0]
            pos = int(fields[1])
            rsid = fields[2]
            ref = fields[3]
            alt_string = fields[4]
            info_string = fields[7]

            info = parse_info(info_string)

            # ------------------------------------------------
            # Candidate missense
            # ------------------------------------------------

            if "NSM" not in info:
                continue

            nsm_records += 1

            chrom = convert_chrom(chrom_raw)

            # ------------------------------------------------
            # ALT alleles
            # ------------------------------------------------

            alts = alt_string.split(",")

            n_alt = len(alts)

            # ------------------------------------------------
            # Clinical annotations
            # ------------------------------------------------

            clnsig_values = get_allele_values(
                info,
                "CLNSIG",
                n_alt
            )

            clnhgvs_values = get_allele_values(
                info,
                "CLNHGVS",
                n_alt
            )

            clndn_values = get_allele_values(
                info,
                "CLNDN",
                n_alt
            )

            clnrevstat_values = get_allele_values(
                info,
                "CLNREVSTAT",
                n_alt
            )

            clnacc_values = get_allele_values(
                info,
                "CLNACC",
                n_alt
            )

            gene = info.get(
                "GENEINFO",
                "."
            )

            # ------------------------------------------------
            # Process each ALT separately
            # ------------------------------------------------

            for i, alt in enumerate(alts):

                allele_index = i + 1

                clnsig = clnsig_values[
                    allele_index
                ]

                label = get_label(clnsig)

                # No usable P/B classification
                if label is None:

                    excluded_count += 1

                    continue

                # ------------------------------------------------
                # Variant ID
                # ------------------------------------------------

                Variant_ID = (
                    f"{chrom}_{pos}_{ref}_{alt}"
                )

                # ------------------------------------------------
                # Other annotations
                # ------------------------------------------------

                clnhgvs = clnhgvs_values[
                    allele_index
                ]

                clndn = clndn_values[
                    allele_index
                ]

                clnrevstat = clnrevstat_values[
                    allele_index
                ]

                clnacc = clnacc_values[
                    allele_index
                ]

                # ------------------------------------------------
                # Add row
                # ------------------------------------------------

                batch.append({

                    "Variant_ID": Variant_ID,

                    "rsid": rsid,

                    "chrom": chrom,

                    "pos": pos,

                    "ref": ref,

                    "alt": alt,

                    "gene": gene,

                    "clnsig": clnsig,

                    "label": label,

                    "clnhgvs": clnhgvs,

                    "clndn": clndn,

                    "clnrevstat": clnrevstat,

                    "clnacc": clnacc,
                })

                output_rows += 1

                # ------------------------------------------------
                # Statistics
                # ------------------------------------------------

                if label == 0:

                    benign_count += 1

                else:

                    pathogenic_count += 1

                chrom_counts[chrom] = (
                    chrom_counts.get(chrom, 0) + 1
                )

                # ------------------------------------------------
                # Write batch
                # ------------------------------------------------

                if len(batch) >= BATCH_SIZE:

                    write_batch()


finally:

    # Write remaining rows
    write_batch()

    writer.close()

In [1]:
import os
import pyarrow.parquet as pq
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

PARQUET_PATH = r"D:\variant_data\dbsnp_missense_raw.parquet"


# ============================================================
# 1. FILE INFORMATION
# ============================================================

print("=" * 70)
print("1. FILE INFORMATION")
print("=" * 70)

print("Exists:", os.path.exists(PARQUET_PATH))

if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(PARQUET_PATH)

file_size_gb = os.path.getsize(PARQUET_PATH) / (1024 ** 3)

print(f"File size: {file_size_gb:.3f} GB")


# ============================================================
# 2. PARQUET METADATA
# ============================================================

print("\n" + "=" * 70)
print("2. PARQUET METADATA")
print("=" * 70)

pf = pq.ParquetFile(PARQUET_PATH)

print("Rows:", f"{pf.metadata.num_rows:,}")
print("Row groups:", f"{pf.num_row_groups:,}")
print("Columns:", pf.schema.names)


# ============================================================
# 3. LOAD ONLY NECESSARY COLUMNS
# ============================================================

columns = [
    "Variant_ID",
    "rsid",
    "chrom",
    "pos",
    "ref",
    "alt",
    "gene",
    "clnsig",
    "label",
    "clnhgvs",
    "clndn",
    "clnrevstat",
    "clnacc"
]

df = pd.read_parquet(
    PARQUET_PATH,
    columns=columns
)

print("\nLoaded dataframe:")
print("Shape:", df.shape)


# ============================================================
# 4. LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("3. LABEL DISTRIBUTION")
print("=" * 70)

label_counts = df["label"].value_counts(dropna=False)

print(label_counts)

print("\nPercentage:")

print(
    (df["label"].value_counts(normalize=True) * 100)
    .round(2)
)


# ============================================================
# 5. CHECK LABEL VALIDITY
# ============================================================

print("\n" + "=" * 70)
print("4. LABEL VALIDITY")
print("=" * 70)

valid_labels = {0, 1}

invalid_labels = df.loc[
    ~df["label"].isin(valid_labels),
    "label"
]

print("Invalid labels:", len(invalid_labels))

if len(invalid_labels) == 0:
    print("✓ All labels are 0/1")
else:
    print(invalid_labels.value_counts())


# ============================================================
# 6. CHECK VARIANT_ID FORMAT
# ============================================================

print("\n" + "=" * 70)
print("5. VARIANT_ID FORMAT")
print("=" * 70)

print(df["Variant_ID"].head(10).to_string(index=False))


# Expected:
# chromosome_position_REF_ALT
#
# Example:
# 1_1014042_G_A

pattern = r"^(1[0-9]|2[0-2]|[1-9]|X|Y)_[0-9]+_[ACGT]+_[ACGT]+$"

valid_variant_id = df["Variant_ID"].astype(str).str.match(
    pattern,
    na=False
)

print("\nValid Variant_ID:", valid_variant_id.sum())
print("Invalid Variant_ID:", (~valid_variant_id).sum())

if (~valid_variant_id).sum() == 0:
    print("✓ Variant_ID format is correct")
else:
    print(
        df.loc[
            ~valid_variant_id,
            "Variant_ID"
        ].head(20)
    )


# ============================================================
# 7. RE-CALCULATE VARIANT_ID
# ============================================================

print("\n" + "=" * 70)
print("6. VARIANT_ID CONSISTENCY")
print("=" * 70)

expected_variant_id = (
    df["chrom"].astype(str)
    + "_"
    + df["pos"].astype(str)
    + "_"
    + df["ref"].astype(str)
    + "_"
    + df["alt"].astype(str)
)

variant_id_match = (
    df["Variant_ID"].astype(str)
    == expected_variant_id
)

print("Correct Variant_ID:", variant_id_match.sum())
print("Incorrect Variant_ID:", (~variant_id_match).sum())

if (~variant_id_match).sum() == 0:
    print("✓ Variant_ID is internally consistent")
else:
    print(
        df.loc[
            ~variant_id_match,
            [
                "Variant_ID",
                "chrom",
                "pos",
                "ref",
                "alt"
            ]
        ].head(20)
    )


# ============================================================
# 8. DUPLICATE VARIANT_ID
# ============================================================

print("\n" + "=" * 70)
print("7. DUPLICATES")
print("=" * 70)

duplicate_variant_id = df["Variant_ID"].duplicated(
    keep=False
)

print(
    "Duplicate rows:",
    duplicate_variant_id.sum()
)

print(
    "Unique Variant_ID:",
    df["Variant_ID"].nunique()
)

if duplicate_variant_id.sum() == 0:
    print("✓ No duplicate Variant_ID")
else:

    print("\nExamples:")

    print(
        df.loc[
            duplicate_variant_id,
            [
                "Variant_ID",
                "rsid",
                "label",
                "clnsig",
                "clnacc"
            ]
        ]
        .sort_values("Variant_ID")
        .head(30)
    )


# ============================================================
# 9. SAME VARIANT WITH DIFFERENT LABELS
# ============================================================

print("\n" + "=" * 70)
print("8. CONFLICTING LABELS FOR SAME VARIANT")
print("=" * 70)

label_per_variant = (
    df.groupby("Variant_ID")["label"]
    .nunique()
)

conflicting_variants = label_per_variant[
    label_per_variant > 1
]

print(
    "Variants with both labels:",
    len(conflicting_variants)
)

if len(conflicting_variants) > 0:

    conflict_ids = conflicting_variants.index

    print(
        df[
            df["Variant_ID"].isin(conflict_ids)
        ][
            [
                "Variant_ID",
                "rsid",
                "label",
                "clnsig",
                "clndn",
                "clnrevstat",
                "clnacc"
            ]
        ]
        .sort_values("Variant_ID")
        .head(50)
    )


# ============================================================
# 10. CHROMOSOME DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("9. CHROMOSOME DISTRIBUTION")
print("=" * 70)

chrom_order = [
    str(i) for i in range(1, 23)
] + ["X", "Y"]

chrom_counts = (
    df["chrom"]
    .value_counts()
    .reindex(chrom_order)
    .dropna()
)

print(chrom_counts)


# ============================================================
# 11. LABEL DISTRIBUTION BY CHROMOSOME
# ============================================================

print("\n" + "=" * 70)
print("10. LABEL BY CHROMOSOME")
print("=" * 70)

chrom_label = pd.crosstab(
    df["chrom"],
    df["label"]
)

chrom_label = chrom_label.reindex(
    chrom_order
).dropna(how="all")

print(chrom_label)


# ============================================================
# 12. NULL / MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("11. MISSING VALUES")
print("=" * 70)

missing = df.isna().sum()

missing_percent = (
    missing / len(df) * 100
).round(3)

missing_table = pd.DataFrame({
    "missing": missing,
    "percentage": missing_percent
})

print(missing_table)


# ============================================================
# 13. BASIC VALIDATION OF REF / ALT
# ============================================================

print("\n" + "=" * 70)
print("12. REF / ALT VALIDATION")
print("=" * 70)

valid_bases = df["ref"].astype(str).str.fullmatch(
    r"[ACGT]+"
) & df["alt"].astype(str).str.fullmatch(
    r"[ACGT]+"
)

print("Valid REF/ALT:", valid_bases.sum())
print("Invalid REF/ALT:", (~valid_bases).sum())

if (~valid_bases).sum() > 0:

    print(
        df.loc[
            ~valid_bases,
            ["Variant_ID", "ref", "alt"]
        ].head(20)
    )


# ============================================================
# 14. POSITION VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("13. POSITION VALIDATION")
print("=" * 70)

print("Minimum position:", df["pos"].min())
print("Maximum position:", df["pos"].max())

print(
    "Invalid positions:",
    (df["pos"] <= 0).sum()
)


# ============================================================
# 15. SAMPLE DATA
# ============================================================

print("\n" + "=" * 70)
print("14. SAMPLE DATA")
print("=" * 70)

print(
    df[
        [
            "Variant_ID",
            "rsid",
            "gene",
            "clnsig",
            "label",
            "clnhgvs",
            "clndn"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"Total rows:          {len(df):,}")
print(f"Unique Variant_ID:   {df['Variant_ID'].nunique():,}")
print(f"Benign/LB:           {(df['label'] == 0).sum():,}")
print(f"Pathogenic/LP:       {(df['label'] == 1).sum():,}")
print(f"Duplicate IDs:       {duplicate_variant_id.sum():,}")
print(f"Conflicting labels:  {len(conflicting_variants):,}")
print(f"Invalid Variant_ID:  {(~valid_variant_id).sum():,}")
print(f"Missing values total:{missing.sum():,}")

print("\n✓ Validation complete.")

1. FILE INFORMATION
Exists: True
File size: 0.010 GB

2. PARQUET METADATA
Rows: 310,305
Row groups: 4
Columns: ['Variant_ID', 'rsid', 'chrom', 'pos', 'ref', 'alt', 'gene', 'clnsig', 'label', 'clnhgvs', 'clndn', 'clnrevstat', 'clnacc']

Loaded dataframe:
Shape: (310305, 13)

3. LABEL DISTRIBUTION
label
0    223534
1     86771
Name: count, dtype: int64

Percentage:
label
0    72.04
1    27.96
Name: proportion, dtype: float64

4. LABEL VALIDITY
Invalid labels: 0
✓ All labels are 0/1

5. VARIANT_ID FORMAT
 1_69134_A_G
1_925969_C_T
1_930165_G_A
1_930204_G_A
1_930220_G_A
1_930245_G_A
1_930248_G_A
1_930256_C_T
1_930259_C_T
1_930265_T_C

Valid Variant_ID: 278683
Invalid Variant_ID: 31622
11432          1_112929212_A_N
72920          4_169508827_A_N
74090            5_1282488_C_N
103186          7_44152363_C_N
118518          8_96145055_C_N
118977         8_102232169_A_N
123232          9_14812983_C_N
128690         9_120568376_A_N
131157         9_132334624_C_N
134954         10_14919809_G_N
1

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


INPUT_PATH = r"D:\variant_data\dbsnp_missense_raw.parquet"

OUTPUT_PATH = r"D:\variant_data\dbsnp_missense_raw.parquet"


# ============================================================
# LOAD
# ============================================================

df = pd.read_parquet(INPUT_PATH)

print("Original shape:", df.shape)


# ============================================================
# 1. KEEP ONLY GRCh38 PRIMARY CHROMOSOMES
# ============================================================

valid_chromosomes = (
    [str(i) for i in range(1, 23)]
    + ["X", "Y"]
)

mask_chrom = df["chrom"].isin(valid_chromosomes)

print(
    "Non-primary chromosomes:",
    (~mask_chrom).sum()
)


# ============================================================
# 2. KEEP ONLY STANDARD DNA BASES
# ============================================================

mask_ref = df["ref"].str.fullmatch(
    r"[ACGT]+",
    na=False
)

mask_alt = df["alt"].str.fullmatch(
    r"[ACGT]+",
    na=False
)

mask_bases = mask_ref & mask_alt

print(
    "Non-standard REF/ALT:",
    (~mask_bases).sum()
)


# ============================================================
# 3. COMBINE FILTERS
# ============================================================

mask = mask_chrom & mask_bases

clean_df = df.loc[mask].copy()


# ============================================================
# 4. RE-CHECK DUPLICATES
# ============================================================

duplicates = clean_df["Variant_ID"].duplicated()

print(
    "Duplicates after cleaning:",
    duplicates.sum()
)

clean_df = clean_df.loc[~duplicates].copy()


# ============================================================
# 5. SAVE PARQUET
# ============================================================

clean_df.to_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# 6. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLEAN DATASET")
print("=" * 70)

print("Output:", OUTPUT_PATH)

print("Shape:", clean_df.shape)

print("\nLabels:")
print(clean_df["label"].value_counts())

print("\nLabel percentage:")
print(
    clean_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nChromosomes:")
print(
    clean_df["chrom"]
    .value_counts()
    .reindex(valid_chromosomes)
    .dropna()
)

print("\nVariant_ID unique:")
print(clean_df["Variant_ID"].nunique())

print("\nMissing:")
print(clean_df.isna().sum().sum())

print("\nDone.")

Original shape: (310305, 13)
Non-primary chromosomes: 31606
Non-standard REF/ALT: 17
Duplicates after cleaning: 0

CLEAN DATASET
Output: D:\variant_data\dbsnp_missense_raw1.parquet
Shape: (278683, 13)

Labels:
label
0    198956
1     79727
Name: count, dtype: int64

Label percentage:
label
0    71.39
1    28.61
Name: proportion, dtype: float64

Chromosomes:
chrom
1     24243
2     26023
3     14717
4      8673
5     13516
6     11520
7     14176
8      9027
9     12476
10     9000
11    15910
12    13190
13     6599
14     8346
15     9490
16    14835
17    17513
18     4660
19    14738
20     5817
21     3349
22     5763
X     14984
Y       118
Name: count, dtype: int64

Variant_ID unique:
278683

Missing:
0

Done.


In [1]:
import pandas as pd

df = pd.read_parquet(r"D:\variant_data\dbsnp_missense_raw.parquet")
df

,Variant_ID,rsid,chrom,pos,ref,alt,gene,clnsig,label,clnhgvs,clndn,clnrevstat,clnacc
0,1_69134_A_G,rs781394307,1,69134,A,G,OR4F5:79501,3,0,NC_000001.11:g.69134A>G,not_specified,single,RCV004071174.1
1,1_925969_C_T,rs200686669,1,925969,C,T,SAMD11:148398|LOC107985728:107985728,3,0,NC_000001.11:g.925969C>T,not_provided,single,RCV002141192.5
2,1_930165_G_A,rs201186828,1,930165,G,A,SAMD11:148398,2|2,0,NC_000001.11:g.930165G>A,not_provided|SAMD11-related_disorder,single|single,RCV001510902.6|RCV003908806.1
3,1_930204_G_A,rs148711625,1,930204,G,A,SAMD11:148398,2|2,0,NC_000001.11:g.930204G>A,not_provided|SAMD11-related_disorder,single|single,RCV001522359.5|RCV003980616.1
4,1_930220_G_A,rs2100306518,1,930220,G,A,SAMD11:148398,3,0,NC_000001.11:g.930220G>A,not_provided,single,RCV002189215.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
278678,Y_57087054_T_C,rs145477164,Y,57087054,T,C,VAMP7:6845,2,0,NC_000024.10:g.57087054T>C,not_provided,single,RCV000895187.4
278679,Y_57126292_T_C,rs141017429,Y,57126292,T,C,VAMP7:6845,3|3,0,NC_000024.10:g.57126292T>C,not_provided|VAMP7-related_disorder,single|single,RCV000880306.4|RCV003895404.1
278680,Y_57126304_C_T,rs138653672,Y,57126304,C,T,VAMP7:6845,3,0,NC_000024.10:g.57126304C>T,not_provided,single,RCV003312790.9
278681,Y_57196354_G_A,rs142104921,Y,57196354,G,A,IL9R:3581,2,0,NC_000024.10:g.57196354G>A,not_provided,single,RCV000948172.3


In [ ]:
import pandas as pd
import numpy as np
import os


# ============================================================
# CONFIG
# ============================================================

CLINVAR_PATH = r"D:\variant_data\clinvar_vep_mapping_final_with_date.parquet"

DBSNP_PATH = r"D:\variant_data\dbsnp_missense_raw.parquet"

OUTPUT_MERGED = r"D:\variant_data\clinvar_dbsnp_merged.parquet"

OUTPUT_DBSNP_UNIQUE = r"D:\variant_data\dbsnp_unique_for_VEP.parquet"

OUTPUT_OVERLAP = r"D:\variant_data\clinvar_dbsnp_overlap.parquet"


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_MERGED),
    exist_ok=True
)


# ============================================================
# 1. LOAD CLINVAR
# ============================================================

print("=" * 70)
print("LOADING CLINVAR")
print("=" * 70)

clinvar = pd.read_parquet(CLINVAR_PATH)

print("ClinVar shape:", clinvar.shape)

print("\nClinVar columns:")
print(clinvar.columns.tolist())


# ============================================================
# 2. LOAD DBSNP
# ============================================================

print("\n" + "=" * 70)
print("LOADING dbSNP")
print("=" * 70)

dbsnp = pd.read_parquet(DBSNP_PATH)

print("dbSNP shape:", dbsnp.shape)

print("\ndbSNP columns:")
print(dbsnp.columns.tolist())


# ============================================================
# 3. STANDARDIZE CLINVAR COLUMN NAMES
# ============================================================

# We DO NOT rename the actual ClinVar dataframe.
# Instead, create a standardized key.

required_clinvar = [
    "Variant_ID",
    "CHROM",
    "POS",
    "REF",
    "ALT",
    "Label"
]

missing = [
    col for col in required_clinvar
    if col not in clinvar.columns
]

if missing:
    raise ValueError(
        f"ClinVar is missing columns: {missing}"
    )


# ============================================================
# 4. STANDARDIZE TYPES
# ============================================================

clinvar["Variant_ID"] = (
    clinvar["Variant_ID"]
    .astype(str)
    .str.strip()
)

clinvar["CHROM"] = (
    clinvar["CHROM"]
    .astype(str)
    .str.strip()
)

clinvar["REF"] = (
    clinvar["REF"]
    .astype(str)
    .str.upper()
    .str.strip()
)

clinvar["ALT"] = (
    clinvar["ALT"]
    .astype(str)
    .str.upper()
    .str.strip()
)

dbsnp["Variant_ID"] = (
    dbsnp["Variant_ID"]
    .astype(str)
    .str.strip()
)

dbsnp["chrom"] = (
    dbsnp["chrom"]
    .astype(str)
    .str.strip()
)

dbsnp["ref"] = (
    dbsnp["ref"]
    .astype(str)
    .str.upper()
    .str.strip()
)

dbsnp["alt"] = (
    dbsnp["alt"]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ============================================================
# 5. REBUILD VARIANT_ID FOR CLINVAR
# ============================================================

clinvar["Variant_ID_check"] = (
    clinvar["CHROM"]
    + "_"
    + clinvar["POS"].astype(str)
    + "_"
    + clinvar["REF"]
    + "_"
    + clinvar["ALT"]
)


# ============================================================
# 6. CHECK CLINVAR Variant_ID CONSISTENCY
# ============================================================

print("\n" + "=" * 70)
print("CHECKING CLINVAR Variant_ID")
print("=" * 70)

variant_id_match = (
    clinvar["Variant_ID"]
    == clinvar["Variant_ID_check"]
)

print(
    "Correct:",
    variant_id_match.sum()
)

print(
    "Incorrect:",
    (~variant_id_match).sum()
)

if (~variant_id_match).sum() > 0:

    print("\nExamples:")

    print(
        clinvar.loc[
            ~variant_id_match,
            [
                "Variant_ID",
                "Variant_ID_check",
                "CHROM",
                "POS",
                "REF",
                "ALT"
            ]
        ].head(20)
    )

    raise ValueError(
        "ClinVar Variant_ID does not consistently match "
        "CHROM/POS/REF/ALT. Stop before merging."
    )


# ============================================================
# 7. CHECK DUPLICATES INSIDE CLINVAR
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DUPLICATES INSIDE CLINVAR")
print("=" * 70)

clinvar_dup = clinvar["Variant_ID"].duplicated(
    keep=False
)

print(
    "Duplicate ClinVar rows:",
    clinvar_dup.sum()
)

print(
    "Unique ClinVar variants:",
    clinvar["Variant_ID"].nunique()
)

if clinvar_dup.sum() > 0:

    print("\nDuplicate examples:")

    print(
        clinvar.loc[
            clinvar_dup,
            [
                "Variant_ID",
                "Label"
            ]
        ]
        .sort_values("Variant_ID")
        .head(30)
    )


# ============================================================
# 8. CHECK DUPLICATES INSIDE dbSNP
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DUPLICATES INSIDE dbSNP")
print("=" * 70)

dbsnp_dup = dbsnp["Variant_ID"].duplicated(
    keep=False
)

print(
    "Duplicate dbSNP rows:",
    dbsnp_dup.sum()
)

print(
    "Unique dbSNP variants:",
    dbsnp["Variant_ID"].nunique()
)

if dbsnp_dup.sum() > 0:

    print("\nDuplicate examples:")

    print(
        dbsnp.loc[
            dbsnp_dup,
            [
                "Variant_ID",
                "label",
                "clnsig"
            ]
        ]
        .sort_values("Variant_ID")
        .head(30)
    )


# ============================================================
# 9. FIND OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("CHECKING CLINVAR ↔ dbSNP OVERLAP")
print("=" * 70)

clinvar_ids = set(
    clinvar["Variant_ID"]
)

dbsnp_ids = set(
    dbsnp["Variant_ID"]
)

overlap_ids = (
    clinvar_ids
    & dbsnp_ids
)

dbsnp_unique_ids = (
    dbsnp_ids
    - clinvar_ids
)

clinvar_only_ids = (
    clinvar_ids
    - dbsnp_ids
)


print(
    "ClinVar variants:",
    f"{len(clinvar_ids):,}"
)

print(
    "dbSNP variants:",
    f"{len(dbsnp_ids):,}"
)

print(
    "Overlap:",
    f"{len(overlap_ids):,}"
)

print(
    "dbSNP unique:",
    f"{len(dbsnp_unique_ids):,}"
)

print(
    "ClinVar only:",
    f"{len(clinvar_only_ids):,}"
)


# ============================================================
# 10. EXTRACT dbSNP UNIQUE
# ============================================================

print("\n" + "=" * 70)
print("CREATING dbSNP UNIQUE DATASET")
print("=" * 70)

dbsnp_unique = dbsnp[
    dbsnp["Variant_ID"].isin(
        dbsnp_unique_ids
    )
].copy()

print(
    "dbSNP unique shape:",
    dbsnp_unique.shape
)


# ============================================================
# 11. EXTRACT OVERLAP
# ============================================================

overlap = dbsnp[
    dbsnp["Variant_ID"].isin(
        overlap_ids
    )
].copy()

print(
    "Overlap shape:",
    overlap.shape
)


# ============================================================
# 12. REMOVE TEMP COLUMN FROM CLINVAR
# ============================================================

clinvar = clinvar.drop(
    columns=["Variant_ID_check"]
)


# ============================================================
# 13. ADD SOURCE COLUMN
# ============================================================

clinvar["Source"] = "ClinVar"

dbsnp_unique["Source"] = "dbSNP"


# ============================================================
# 14. STANDARDIZE LABEL COLUMN
# ============================================================

# ClinVar:
#     Label
#
# dbSNP:
#     label
#
# We want final dataset:
#
#     Label


dbsnp_unique["Label"] = (
    dbsnp_unique["label"]
)

dbsnp_unique = dbsnp_unique.drop(
    columns=["label"]
)


# ============================================================
# 15. ALIGN COLUMNS
# ============================================================

# Keep every ClinVar column.
#
# Any columns that exist only in dbSNP
# will also be retained.

all_columns = list(
    dict.fromkeys(
        list(clinvar.columns)
        + list(dbsnp_unique.columns)
    )
)


clinvar_aligned = clinvar.reindex(
    columns=all_columns
)

dbsnp_aligned = dbsnp_unique.reindex(
    columns=all_columns
)


# ============================================================
# 16. MERGE
# ============================================================

print("\n" + "=" * 70)
print("MERGING")
print("=" * 70)

merged = pd.concat(
    [
        clinvar_aligned,
        dbsnp_aligned
    ],
    ignore_index=True
)


# ============================================================
# 17. FINAL DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DUPLICATE CHECK")
print("=" * 70)

final_dup = merged[
    "Variant_ID"
].duplicated(
    keep=False
)

print(
    "Duplicate Variant_ID:",
    final_dup.sum()
)

if final_dup.sum() > 0:

    print(
        merged.loc[
            final_dup,
            [
                "Variant_ID",
                "Label",
                "Source"
            ]
        ]
        .sort_values("Variant_ID")
        .head(50)
    )

    raise ValueError(
        "Final dataset contains duplicate Variant_ID."
    )


# ============================================================
# 18. SAVE MERGED DATASET
# ============================================================

print("\nSaving merged dataset...")

merged.to_parquet(
    OUTPUT_MERGED,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# 19. SAVE dbSNP UNIQUE DATASET
# ============================================================

print("Saving dbSNP unique dataset...")

dbsnp_unique.to_parquet(
    OUTPUT_DBSNP_UNIQUE,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# 20. SAVE OVERLAP
# ============================================================

print("Saving overlap dataset...")

overlap.to_parquet(
    OUTPUT_OVERLAP,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(
    f"ClinVar:          {len(clinvar):,}"
)

print(
    f"dbSNP:            {len(dbsnp):,}"
)

print(
    f"Overlap:          {len(overlap):,}"
)

print(
    f"dbSNP unique:     {len(dbsnp_unique):,}"
)

print(
    f"Final merged:     {len(merged):,}"
)

print(
    f"Expected merged:  "
    f"{len(clinvar) + len(dbsnp_unique):,}"
)

print("\nFiles:")

print(
    "Merged:",
    OUTPUT_MERGED
)

print(
    "dbSNP unique for VEP:",
    OUTPUT_DBSNP_UNIQUE
)

print(
    "Overlap:",
    OUTPUT_OVERLAP
)

print("\nSources:")
print(
    merged["Source"].value_counts()
)

print("\nLabels:")
print(
    merged["Label"].value_counts(dropna=False)
)

print("\n✓ Merge completed successfully.")

LOADING CLINVAR
ClinVar shape: (110020, 55)

ClinVar columns:
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'CHROM', 'Start', 'Stop', 'ReviewStatus', 'VariationID', 'POS', 'REF', 'ALT', 'SomaticClinicalImpact', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'ReviewStatusOncogenicity', 'Location', 'Allele', 'Gene', 'SYMBOL', 'Feature', 'Feature_type', 'Consequence', 'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids', 'Codons', 'MANE_SELECT', 'CANONICAL', 'AF', 'gnomADe_AF', 'SpliceAI_pred', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals', 'Variant_ID', 'LastEvaluated', 'Label']

LOADING dbSNP
dbSNP shape: (278683, 13)

dbSNP columns:
['Variant_ID', 'rsid', 'chrom', '

In [4]:
pd.read_parquet(r'D:\variant_data\dbsnp_unique_for_VEP.parquet')

,Variant_ID,rsid,chrom,pos,ref,alt,gene,clnsig,clnhgvs,clndn,clnrevstat,clnacc,Source,Label
0,1_69134_A_G,rs781394307,1,69134,A,G,OR4F5:79501,3,NC_000001.11:g.69134A>G,not_specified,single,RCV004071174.1,dbSNP,0
1,1_925969_C_T,rs200686669,1,925969,C,T,SAMD11:148398|LOC107985728:107985728,3,NC_000001.11:g.925969C>T,not_provided,single,RCV002141192.5,dbSNP,0
2,1_930165_G_A,rs201186828,1,930165,G,A,SAMD11:148398,2|2,NC_000001.11:g.930165G>A,not_provided|SAMD11-related_disorder,single|single,RCV001510902.6|RCV003908806.1,dbSNP,0
3,1_930204_G_A,rs148711625,1,930204,G,A,SAMD11:148398,2|2,NC_000001.11:g.930204G>A,not_provided|SAMD11-related_disorder,single|single,RCV001522359.5|RCV003980616.1,dbSNP,0
4,1_930220_G_A,rs2100306518,1,930220,G,A,SAMD11:148398,3,NC_000001.11:g.930220G>A,not_provided,single,RCV002189215.5,dbSNP,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201928,Y_57087054_T_C,rs145477164,Y,57087054,T,C,VAMP7:6845,2,NC_000024.10:g.57087054T>C,not_provided,single,RCV000895187.4,dbSNP,0
201929,Y_57126292_T_C,rs141017429,Y,57126292,T,C,VAMP7:6845,3|3,NC_000024.10:g.57126292T>C,not_provided|VAMP7-related_disorder,single|single,RCV000880306.4|RCV003895404.1,dbSNP,0
201930,Y_57126304_C_T,rs138653672,Y,57126304,C,T,VAMP7:6845,3,NC_000024.10:g.57126304C>T,not_provided,single,RCV003312790.9,dbSNP,0
201931,Y_57196354_G_A,rs142104921,Y,57196354,G,A,IL9R:3581,2,NC_000024.10:g.57196354G>A,not_provided,single,RCV000948172.3,dbSNP,0


In [1]:
import pandas as pd

CLINVAR_PATH = r"D:\variant_data\clinvar_vep_mapping_final_with_date.parquet"

OVERLAP_PATH = r"D:\variant_data\clinvar_dbsnp_overlap.parquet"


# ============================================================
# LOAD
# ============================================================

clinvar = pd.read_parquet(CLINVAR_PATH)

overlap = pd.read_parquet(OVERLAP_PATH)

print("ClinVar:", clinvar.shape)
print("Overlap:", overlap.shape)


# ============================================================
# KEEP ONLY WHAT WE NEED FROM CLINVAR
# ============================================================

clinvar_labels = clinvar[
    [
        "Variant_ID",
        "Label"
    ]
].copy()


# ============================================================
# MERGE LABELS
# ============================================================

check = overlap.merge(
    clinvar_labels,
    on="Variant_ID",
    how="left",
    suffixes=("_dbSNP", "_ClinVar")
)


# ============================================================
# CHECK
# ============================================================

print("\n" + "=" * 70)
print("LABEL AGREEMENT")
print("=" * 70)

same = (
    check["label"]
    == check["Label"]
)

print(
    "Same label:",
    same.sum()
)

print(
    "Conflicting label:",
    (~same).sum()
)


# ============================================================
# CONTINGENCY TABLE
# ============================================================

print("\n" + "=" * 70)
print("LABEL CROSS-TAB")
print("=" * 70)

print(
    pd.crosstab(
        check["Label"],
        check["label"],
        rownames=["ClinVar"],
        colnames=["dbSNP"]
    )
)


# ============================================================
# CONFLICTING VARIANTS
# ============================================================

conflicts = check.loc[
    ~same
].copy()

print("\n" + "=" * 70)
print("CONFLICTING VARIANTS")
print("=" * 70)

print(
    "Number:",
    len(conflicts)
)

if len(conflicts) > 0:

    print(
        conflicts[
            [
                "Variant_ID",
                "rsid",
                "label",
                "Label",
                "clnsig",
                "clndn"
            ]
        ].head(30)
    )

ClinVar: (110020, 55)
Overlap: (76750, 13)

LABEL AGREEMENT
Same label: 76741
Conflicting label: 9

LABEL CROSS-TAB
dbSNP        0      1
ClinVar              
0        63186      8
1            1  13555

CONFLICTING VARIANTS
Number: 9
            Variant_ID          rsid  label  Label clnsig  \
1447    1_25303329_T_G   rs121912763      1      0      5   
4467   1_159205704_C_T    rs34599082      1      0      5   
21159   5_36971948_G_A  rs1064794954      1      0      4   
23206  5_136054857_G_A   rs376761086      1      0      4   
27046  6_137879193_G_A  rs2114504766      0      1      3   
36846  9_124500196_C_A   rs104894118      1      0      5   
51128  14_53951945_T_C   rs121912765      1      0      5   
70861  22_36285878_T_C   rs764139009      1      0      5   
76740  X_155022432_C_A   rs137852379      1      0    5|5   

                                                   clndn  
1447                           Rhd\x2c_weak_d\x2c_type_I  
4467      DUFFY_BLOOD_GROUP_SYSTEM\

In [1]:
import pandas as pd
import os


# ============================================================
# PATHS
# ============================================================

DBSNP_UNIQUE_PATH = (
    r"D:\variant_data\dbsnp_unique_for_VEP.parquet"
)

MERGED_PATH = (
    r"D:\variant_data\clinvar_dbsnp_merged.parquet"
)

DBSNP_RAW_PATH = (
    r"D:\variant_data\dbsnp_missense_raw.parquet"
)


# Output
DBSNP_UNIQUE_OUT = (
    r"D:\variant_data\dbsnp_unique_for_VEP_standardized.parquet"
)

MERGED_OUT = (
    r"D:\variant_data\clinvar_dbsnp_merged_standardized.parquet"
)

DBSNP_RAW_OUT = (
    r"D:\variant_data\dbsnp_missense_raw_standardized.parquet"
)


# ============================================================
# FUNCTION
# ============================================================

def standardize_variant_columns(
    input_path,
    output_path,
    name
):
    
    print("\n" + "=" * 70)
    print(f"PROCESSING: {name}")
    print("=" * 70)

    df = pd.read_parquet(input_path)

    print("Original shape:", df.shape)

    print("\nOriginal columns:")
    print(df.columns.tolist())


    # --------------------------------------------------------
    # 1. CHECK AVAILABLE COLUMN SET
    # --------------------------------------------------------

    upper_cols = ["CHROM", "POS", "REF", "ALT", "Label"]
    lower_cols = ["chrom", "pos", "ref", "alt", "label"]


    # --------------------------------------------------------
    # 2. HANDLE EACH VARIANT COLUMN
    # --------------------------------------------------------

    # CHROM
    if "CHROM" in df.columns and "chrom" in df.columns:

        # Check consistency before deleting lowercase
        mismatch = (
            df["CHROM"].astype(str).str.strip()
            !=
            df["chrom"].astype(str).str.strip()
        )

        print(
            f"\nCHROM mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:
            print(
                df.loc[
                    mismatch,
                    ["CHROM", "chrom"]
                ].head(20)
            )

            raise ValueError(
                f"{name}: CHROM and chrom contain "
                "different values!"
            )

        df = df.drop(columns=["chrom"])


    elif "CHROM" not in df.columns and "chrom" in df.columns:

        df = df.rename(
            columns={"chrom": "CHROM"}
        )


    # POS
    if "POS" in df.columns and "pos" in df.columns:

        mismatch = (
            pd.to_numeric(
                df["POS"],
                errors="coerce"
            )
            !=
            pd.to_numeric(
                df["pos"],
                errors="coerce"
            )
        )

        print(
            f"POS mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:
            print(
                df.loc[
                    mismatch,
                    ["POS", "pos"]
                ].head(20)
            )

            raise ValueError(
                f"{name}: POS and pos contain "
                "different values!"
            )

        df = df.drop(columns=["pos"])


    elif "POS" not in df.columns and "pos" in df.columns:

        df = df.rename(
            columns={"pos": "POS"}
        )


    # REF
    if "REF" in df.columns and "ref" in df.columns:

        mismatch = (
            df["REF"].astype(str).str.upper().str.strip()
            !=
            df["ref"].astype(str).str.upper().str.strip()
        )

        print(
            f"REF mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:
            print(
                df.loc[
                    mismatch,
                    ["REF", "ref"]
                ].head(20)
            )

            raise ValueError(
                f"{name}: REF and ref contain "
                "different values!"
            )

        df = df.drop(columns=["ref"])


    elif "REF" not in df.columns and "ref" in df.columns:

        df = df.rename(
            columns={"ref": "REF"}
        )


    # ALT
    if "ALT" in df.columns and "alt" in df.columns:

        mismatch = (
            df["ALT"].astype(str).str.upper().str.strip()
            !=
            df["alt"].astype(str).str.upper().str.strip()
        )

        print(
            f"ALT mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:
            print(
                df.loc[
                    mismatch,
                    ["ALT", "alt"]
                ].head(20)
            )

            raise ValueError(
                f"{name}: ALT and alt contain "
                "different values!"
            )

        df = df.drop(columns=["alt"])


    elif "ALT" not in df.columns and "alt" in df.columns:

        df = df.rename(
            columns={"alt": "ALT"}
        )


    # Label
    if "Label" in df.columns and "label" in df.columns:

        # Convert both to numeric before comparing
        upper_label = pd.to_numeric(
            df["Label"],
            errors="coerce"
        )

        lower_label = pd.to_numeric(
            df["label"],
            errors="coerce"
        )

        mismatch = (
            upper_label != lower_label
        )

        # Ignore rows where both are NaN
        mismatch = (
            mismatch
            &
            ~(upper_label.isna() & lower_label.isna())
        )

        print(
            f"Label mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:
            print(
                df.loc[
                    mismatch,
                    ["Label", "label"]
                ].head(20)
            )

            raise ValueError(
                f"{name}: Label and label contain "
                "different values!"
            )

        df = df.drop(columns=["label"])


    elif "Label" not in df.columns and "label" in df.columns:

        df = df.rename(
            columns={"label": "Label"}
        )


    # --------------------------------------------------------
    # 3. REQUIRED COLUMNS CHECK
    # --------------------------------------------------------

    required = [
        "CHROM",
        "POS",
        "REF",
        "ALT",
        "Label"
    ]

    missing = [
        col for col in required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{name}: Missing required columns: {missing}"
        )


    # --------------------------------------------------------
    # 4. STANDARDIZE DATA TYPES
    # --------------------------------------------------------

    df["CHROM"] = (
        df["CHROM"]
        .astype(str)
        .str.strip()
    )

    df["POS"] = pd.to_numeric(
        df["POS"],
        errors="raise"
    ).astype("int64")

    df["REF"] = (
        df["REF"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["ALT"] = (
        df["ALT"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["Label"] = pd.to_numeric(
        df["Label"],
        errors="raise"
    ).astype("int8")


    # --------------------------------------------------------
    # 5. CHECK LABEL
    # --------------------------------------------------------

    invalid_labels = ~df["Label"].isin([0, 1])

    print(
        "\nInvalid labels:",
        invalid_labels.sum()
    )

    if invalid_labels.sum() > 0:

        print(
            df.loc[
                invalid_labels,
                ["Variant_ID", "Label"]
            ].head(20)
        )

        raise ValueError(
            f"{name}: Invalid Label values detected!"
        )


    # --------------------------------------------------------
    # 6. CHECK VARIANT_ID CONSISTENCY
    # --------------------------------------------------------

    if "Variant_ID" in df.columns:

        expected_id = (
            df["CHROM"].astype(str)
            + "_"
            + df["POS"].astype(str)
            + "_"
            + df["REF"]
            + "_"
            + df["ALT"]
        )

        mismatch = (
            df["Variant_ID"].astype(str)
            != expected_id
        )

        print(
            "Variant_ID mismatches:",
            mismatch.sum()
        )

        if mismatch.sum() > 0:

            print(
                df.loc[
                    mismatch,
                    [
                        "Variant_ID",
                        "CHROM",
                        "POS",
                        "REF",
                        "ALT"
                    ]
                ].head(20)
            )

            raise ValueError(
                f"{name}: Variant_ID is inconsistent "
                "with CHROM/POS/REF/ALT!"
            )


    # --------------------------------------------------------
    # 7. CHECK DUPLICATES
    # --------------------------------------------------------

    if "Variant_ID" in df.columns:

        duplicates = df["Variant_ID"].duplicated(
            keep=False
        )

        print(
            "Duplicate Variant_ID rows:",
            duplicates.sum()
        )


    # --------------------------------------------------------
    # 8. SAVE
    # --------------------------------------------------------

    df.to_parquet(
        output_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )


    # --------------------------------------------------------
    # 9. FINAL REPORT
    # --------------------------------------------------------

    print("\nFinal shape:", df.shape)

    print("\nFinal variant columns:")

    for col in required:
        print(
            f"{col:8s} -> {df[col].dtype}"
        )

    print("\nFinal columns:")
    print(df.columns.tolist())

    print("\nSaved:")
    print(output_path)

    return df


# ============================================================
# PROCESS 1
# ============================================================

dbsnp_unique = standardize_variant_columns(
    DBSNP_UNIQUE_PATH,
    DBSNP_UNIQUE_OUT,
    "dbSNP UNIQUE FOR VEP"
)


# ============================================================
# PROCESS 2
# ============================================================

# merged = standardize_variant_columns(
#     MERGED_PATH,
#     MERGED_OUT,
#     "CLINVAR + dbSNP MERGED"
# )


# ============================================================
# PROCESS 3
# ============================================================

dbsnp_raw = standardize_variant_columns(
    DBSNP_RAW_PATH,
    DBSNP_RAW_OUT,
    "dbSNP MISSENSE RAW"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ALL FILES STANDARDIZED SUCCESSFULLY")
print("=" * 70)

print(
    "\n1. dbSNP unique:"
    f"\n   {DBSNP_UNIQUE_OUT}"
)

# print(
#     "\n2. ClinVar + dbSNP:"
#     f"\n   {MERGED_OUT}"
# )

print(
    "\n3. dbSNP raw:"
    f"\n   {DBSNP_RAW_OUT}"
)


PROCESSING: dbSNP UNIQUE FOR VEP
Original shape: (201933, 14)

Original columns:
['Variant_ID', 'rsid', 'chrom', 'pos', 'ref', 'alt', 'gene', 'clnsig', 'clnhgvs', 'clndn', 'clnrevstat', 'clnacc', 'Source', 'Label']

Invalid labels: 0
Variant_ID mismatches: 0
Duplicate Variant_ID rows: 0

Final shape: (201933, 14)

Final variant columns:
CHROM    -> str
POS      -> int64
REF      -> str
ALT      -> str
Label    -> int8

Final columns:
['Variant_ID', 'rsid', 'CHROM', 'POS', 'REF', 'ALT', 'gene', 'clnsig', 'clnhgvs', 'clndn', 'clnrevstat', 'clnacc', 'Source', 'Label']

Saved:
D:\variant_data\dbsnp_unique_for_VEP_standardized.parquet

PROCESSING: dbSNP MISSENSE RAW
Original shape: (278683, 13)

Original columns:
['Variant_ID', 'rsid', 'chrom', 'pos', 'ref', 'alt', 'gene', 'clnsig', 'label', 'clnhgvs', 'clndn', 'clnrevstat', 'clnacc']

Invalid labels: 0
Variant_ID mismatches: 0
Duplicate Variant_ID rows: 0

Final shape: (278683, 13)

Final variant columns:
CHROM    -> str
POS      -> int64

In [1]:
import pandas as pd

INPUT = r"D:\variant_data\clinvar_dbsnp_merged.parquet"

OUTPUT = r"D:\variant_data\clinvar_dbsnp_merged_standardized.parquet"


# ============================================================
# LOAD
# ============================================================

print("=" * 70)
print("LOADING")
print("=" * 70)

df = pd.read_parquet(INPUT)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 1. CHECK COLUMN AVAILABILITY
# ============================================================

upper = {
    "CHROM": "CHROM" in df.columns,
    "POS": "POS" in df.columns,
    "REF": "REF" in df.columns,
    "ALT": "ALT" in df.columns,
    "Label": "Label" in df.columns
}

lower = {
    "chrom": "chrom" in df.columns,
    "pos": "pos" in df.columns,
    "ref": "ref" in df.columns,
    "alt": "alt" in df.columns,
    "label": "label" in df.columns
}

print("\nUppercase:")
print(upper)

print("\nLowercase:")
print(lower)


# ============================================================
# 2. MAKE SURE REQUIRED COLUMNS EXIST
# ============================================================

# For merged file, uppercase ClinVar columns should exist.
required_upper = ["CHROM", "POS", "REF", "ALT", "Label"]

missing = [
    c for c in required_upper
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required uppercase columns: {missing}"
    )


# ============================================================
# 3. IF LOWERCASE COLUMNS EXIST,
#    CHECK ONLY NON-NULL OVERLAP
# ============================================================

pairs = [
    ("CHROM", "chrom"),
    ("POS", "pos"),
    ("REF", "ref"),
    ("ALT", "alt"),
    ("Label", "label")
]

print("\n" + "=" * 70)
print("CHECKING LOWERCASE COLUMNS")
print("=" * 70)

for upper_col, lower_col in pairs:

    if lower_col not in df.columns:
        print(
            f"{lower_col}: NOT FOUND -> skip"
        )
        continue

    # Rows where both columns have data
    mask = (
        df[upper_col].notna()
        &
        df[lower_col].notna()
    )

    print(
        f"\n{upper_col} vs {lower_col}"
    )

    print(
        "Rows where both exist:",
        mask.sum()
    )

    if mask.sum() == 0:
        continue

    # Normalize for comparison
    if upper_col == "POS":

        a = pd.to_numeric(
            df.loc[mask, upper_col],
            errors="coerce"
        )

        b = pd.to_numeric(
            df.loc[mask, lower_col],
            errors="coerce"
        )

    elif upper_col == "Label":

        a = pd.to_numeric(
            df.loc[mask, upper_col],
            errors="coerce"
        )

        b = pd.to_numeric(
            df.loc[mask, lower_col],
            errors="coerce"
        )

    else:

        a = (
            df.loc[mask, upper_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        b = (
            df.loc[mask, lower_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

    mismatch = a != b

    print(
        "Mismatch:",
        mismatch.sum()
    )

    if mismatch.sum() > 0:

        idx = df.loc[mask].index[mismatch]

        print(
            df.loc[
                idx,
                [
                    "Variant_ID",
                    upper_col,
                    lower_col
                ]
            ].head(20)
        )

        raise ValueError(
            f"Conflict detected: "
            f"{upper_col} != {lower_col}"
        )


# ============================================================
# 4. FILL UPPERCASE FROM LOWERCASE ONLY IF NEEDED
# ============================================================

print("\n" + "=" * 70)
print("FILLING MISSING VALUES")
print("=" * 70)

for upper_col, lower_col in pairs:

    if lower_col not in df.columns:
        continue

    missing_upper = df[upper_col].isna()

    print(
        f"{upper_col}:",
        missing_upper.sum(),
        "missing before fill"
    )

    df.loc[missing_upper, upper_col] = (
        df.loc[missing_upper, lower_col]
    )


# ============================================================
# 5. REMOVE LOWERCASE COLUMNS
# ============================================================

lowercase_to_remove = [
    c for c in
    ["chrom", "pos", "ref", "alt", "label"]
    if c in df.columns
]

print("\nRemoving lowercase columns:")
print(lowercase_to_remove)

if lowercase_to_remove:
    df = df.drop(
        columns=lowercase_to_remove
    )


# ============================================================
# 6. STANDARDIZE TYPES
# ============================================================

df["CHROM"] = (
    df["CHROM"]
    .astype(str)
    .str.strip()
)

df["POS"] = pd.to_numeric(
    df["POS"],
    errors="raise"
).astype("int64")

df["REF"] = (
    df["REF"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df["ALT"] = (
    df["ALT"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df["Label"] = pd.to_numeric(
    df["Label"],
    errors="raise"
).astype("int8")


# ============================================================
# 7. LABEL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("LABEL VALIDATION")
print("=" * 70)

print(df["Label"].value_counts())

invalid = ~df["Label"].isin([0, 1])

print(
    "Invalid:",
    invalid.sum()
)

if invalid.sum() > 0:

    print(
        df.loc[
            invalid,
            ["Variant_ID", "Label"]
        ].head(20)
    )

    raise ValueError(
        "Invalid Label values!"
    )


# ============================================================
# 8. VARIANT_ID VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("VARIANT_ID VALIDATION")
print("=" * 70)

expected_id = (
    df["CHROM"].astype(str)
    + "_"
    + df["POS"].astype(str)
    + "_"
    + df["REF"]
    + "_"
    + df["ALT"]
)

mismatch = (
    df["Variant_ID"].astype(str)
    != expected_id
)

print(
    "Variant_ID mismatches:",
    mismatch.sum()
)

if mismatch.sum() > 0:

    print(
        df.loc[
            mismatch,
            [
                "Variant_ID",
                "CHROM",
                "POS",
                "REF",
                "ALT"
            ]
        ].head(20)
    )

    raise ValueError(
        "Variant_ID mismatch!"
    )


# ============================================================
# 9. DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicates = df["Variant_ID"].duplicated(
    keep=False
)

print(
    "Duplicate rows:",
    duplicates.sum()
)

print(
    "Unique Variant_ID:",
    df["Variant_ID"].nunique()
)

if duplicates.sum() > 0:

    print(
        df.loc[
            duplicates
        ].sort_values("Variant_ID").head(20)
    )

    raise ValueError(
        "Duplicate Variant_ID detected!"
    )


# ============================================================
# 10. MISSING VALUES IN CANONICAL COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

canonical = [
    "Variant_ID",
    "CHROM",
    "POS",
    "REF",
    "ALT",
    "Label"
]

print(
    df[canonical].isna().sum()
)


# ============================================================
# 11. FINAL COLUMN CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL COLUMNS")
print("=" * 70)

print(df.columns.tolist())


# ============================================================
# 12. SAVE
# ============================================================

df.to_parquet(
    OUTPUT,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("SUCCESS")
print("=" * 70)

print(
    "Final shape:",
    df.shape
)

print(
    "Saved:",
    OUTPUT
)

print("\nCanonical columns:")

print(
    df[
        [
            "Variant_ID",
            "CHROM",
            "POS",
            "REF",
            "ALT",
            "Label"
        ]
    ].head(10)
)

LOADING
Shape: (311953, 67)

Columns:
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'CHROM', 'Start', 'Stop', 'ReviewStatus', 'VariationID', 'POS', 'REF', 'ALT', 'SomaticClinicalImpact', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'ReviewStatusOncogenicity', 'Location', 'Allele', 'Gene', 'SYMBOL', 'Feature', 'Feature_type', 'Consequence', 'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids', 'Codons', 'MANE_SELECT', 'CANONICAL', 'AF', 'gnomADe_AF', 'SpliceAI_pred', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals', 'Variant_ID', 'LastEvaluated', 'Label', 'Source', 'rsid', 'chrom', 'pos', 'ref', 'alt', 'gene', 'clnsig', 'clnhgvs', 'clndn', 'clnrevstat', 'clnacc']

Upp

In [4]:
pd.read_parquet(r'D:\variant_data\clinvar_dbsnp_merged.parquet')[['CHROM', 'POS', 'REF', 'ALT', 'Label']].head(-10)

,CHROM,POS,REF,ALT,Label
0,1,1014042,G,A,0
1,1,1014228,G,A,0
2,1,1014471,G,C,0
3,1,1020183,G,C,0
4,1,1035307,C,T,0
...,...,...,...,...,...
311938,Y,2787426,C,G,1
311939,Y,2787459,C,A,1
311940,Y,2787515,C,A,1
311941,Y,2787551,C,T,1
